# Mmvlm4SCD on Google Colab — West Africa–informed experiment

This notebook mirrors ``src/scripts/run_full_experiment.py`` (train → evaluate) but seeds the **synthetic** cohort generator with **live** sickle-variant reference frequencies from **1000 Genomes Phase 3 West African panels** (YRI, ESN, GWD, MSL) via the Ensembl Variation REST API for **rs334** (HBB Glu7Val).

**Important:** this is population-level genetics metadata, not individual patient records from Ghana, Nigeria, or other clinical sites (those remain DUA-gated in ``data/loaders``). The multimodal tensors are still simulated; only the genotype mix is tilted using open West African allele-frequency data.

**Citation:** rs334 frequencies from [Ensembl REST](https://rest.ensembl.org/documentation/info/variation_id); populations from [1000 Genomes Phase 3](https://www.internationalgenome.org/).

## 1. Environment setup (Colab or local Jupyter)

- On **Colab**: set ``MMVLM_REPO_URL`` to your fork (or edit ``REPO_URL`` below), then run the next cell.
- **Local**: skip the clone/install block and ensure the repo root is on ``PYTHONPATH`` (e.g. ``pip install -e .`` from the project root).

In [ ]:
import os
import subprocess
import sys

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

ROOT = os.getcwd()
if _in_colab():
    REPO_URL = os.environ.get(
        "MMVLM_REPO_URL",
        "https://github.com/yourusername/Mmvlm4SCD.git",
    )
    DEST = "/content/Mmvlm4SCD"
    if not os.path.isdir(os.path.join(DEST, "src", "mmvlm4scd")):
        subprocess.check_call(
            ["git", "clone", "--depth", "1", REPO_URL, DEST],
            stdout=subprocess.DEVNULL,
        )
    os.chdir(DEST)
    ROOT = DEST
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
else:
    # Local: assume cwd is repo root or parent of notebooks/
    cand = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(cand, "src", "mmvlm4scd")):
        ROOT = cand
        os.chdir(ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

sys.path.insert(0, os.path.join(ROOT, "src"))
print("Repo root:", ROOT)

## 2. Load live West African rs334 frequencies

Requires outbound HTTPS from the runtime (works on default Colab).

In [ ]:
import pandas as pd

from mmvlm4scd.data import (
    fetch_rs334_west_africa_1000g_phase3,
    mean_rs334_maf_west_africa,
)

freq_tbl = fetch_rs334_west_africa_1000g_phase3()
display(freq_tbl)

mean_maf = mean_rs334_maf_west_africa(freq_tbl)
print(f"Mean rs334 MAF (West African 1KG Phase 3 panels): {mean_maf:.4f}")

## 3. Build cohort, train, and evaluate

Same components as ``run_full_experiment.py``: ``StandardPreprocessor``, ``make_loaders``, ``MultimodalSCDModel``, ``Trainer``, ``evaluate_model_full``. Reduce ``n_patients`` / ``epochs`` on CPU; use **Runtime → GPU** on Colab for faster training.

In [ ]:
from pathlib import Path

import numpy as np
import torch

from mmvlm4scd.data import StandardPreprocessor, generate_synthetic_cohort
from mmvlm4scd.data.dataloaders import make_loaders
from mmvlm4scd.data.synthetic import SCDSyntheticConfig, split_indices
from mmvlm4scd.evaluation import evaluate_model_full
from mmvlm4scd.models import ModelConfig, MultimodalSCDModel
from mmvlm4scd.training import Trainer, TrainConfig
from mmvlm4scd.utils import auto_device, set_seed

cfg = {
    "data": {"n_patients": 2000, "timesteps": 24, "seed": 7},
    "model": {
        "embed_dim": 64,
        "fusion": "attention",
        "dropout": 0.1,
        "num_severity_classes": 3,
    },
    "train": {
        "epochs": 12,
        "batch_size": 64,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "grad_clip": 1.0,
        "alpha": 1.0,
        "beta": 0.5,
        "early_stop_patience": 6,
        "select_metric": "auroc_ovr",
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "seed": 0,
    },
}

set_seed(cfg["train"]["seed"])

cohort = generate_synthetic_cohort(
    SCDSyntheticConfig(
        n_patients=cfg["data"]["n_patients"],
        timesteps=cfg["data"]["timesteps"],
        seed=cfg["data"]["seed"],
        west_africa_rs334_maf=mean_maf,
    )
)

pre = StandardPreprocessor().fit(cohort["clinical"])
clin_x = pre.transform(cohort["clinical"])
tr_idx, va_idx, te_idx = split_indices(
    len(cohort["severity"]), seed=cfg["data"]["seed"]
)

tr_loader, va_loader, te_loader = make_loaders(
    cohort,
    clin_x,
    tr_idx,
    va_idx,
    te_idx,
    batch_size=cfg["train"]["batch_size"],
)

device = cfg["train"].get("device") or auto_device()
model = MultimodalSCDModel(
    ModelConfig(
        clinical_input_dim=clin_x.shape[1],
        genomic_input_dim=cohort["genomic"].shape[1],
        imaging_input_dim=cohort["imaging"].shape[1],
        temporal_input_dim=cohort["temporal"].shape[2],
        embed_dim=cfg["model"]["embed_dim"],
        fusion=cfg["model"]["fusion"],
        dropout=cfg["model"]["dropout"],
        num_severity_classes=cfg["model"]["num_severity_classes"],
    )
)

trainer = Trainer(
    model,
    TrainConfig(
        epochs=cfg["train"]["epochs"],
        lr=cfg["train"]["lr"],
        weight_decay=cfg["train"]["weight_decay"],
        grad_clip=cfg["train"]["grad_clip"],
        alpha=cfg["train"]["alpha"],
        beta=cfg["train"]["beta"],
        early_stop_patience=cfg["train"]["early_stop_patience"],
        select_metric=cfg["train"]["select_metric"],
        device=device,
    ),
)

train_out = trainer.fit(tr_loader, va_loader)
print("Best val score:", train_out["best_score"])

ev = evaluate_model_full(model, te_loader, device=device)
metrics = ev["metrics"]
for k in sorted(metrics.keys()):
    v = metrics[k]
    if isinstance(v, (float, int, np.floating, np.integer)):
        print(f"{k}: {float(v):.4f}")

out_dir = Path(ROOT) / "experiments" / "results" / "colab_west_africa"
out_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), out_dir / "last_model.pt")
print("Saved checkpoint to", out_dir / "last_model.pt")